# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- Use Pandas to load, inspect, and clean the dataset appropriately. 
- Transform relevant columns to create measures that address the problem at hand.
- **conduct EDA: visualization and statistical measures to understand the structure of the data**
- **recommend a set of manufacturers to consider as well as specific airplanes conforming to the client's request**
- **discuss the relationship between serious injuries/airplane damage incurred and at least *two* factors at play in the incident. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.**

In [2]:
# loading relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Exploratory Data Analysis  
- Load in the cleaned data

In [ ]:
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid', palette='Set2')

clean_path = Path('data/aviation_cleaned_for_analysis.csv')
raw_path = Path('data/AviationData.csv')

if clean_path.exists():
    aviation = pd.read_csv(clean_path, low_memory=False)
    aviation['Event.Date'] = pd.to_datetime(aviation['Event.Date'], errors='coerce')
else:
    # Rebuild the cleaned analysis fields from the raw data if the cleaning notebook
    # has not yet been run in this workspace.
    aviation = pd.read_csv(raw_path, encoding='latin-1', low_memory=False)
    aviation['Event.Date'] = pd.to_datetime(aviation['Event.Date'], errors='coerce')
    aviation['Make'] = aviation['Make'].astype(str).str.strip()
    aviation['Model'] = aviation['Model'].astype(str).str.strip()
    aviation = aviation[aviation['Event.Date'] >= '1983-01-01'].copy()
    aviation = aviation[aviation['Investigation.Type'] == 'Accident'].copy()
    aviation = aviation[aviation['Amateur.Built'].astype(str).str.strip().str.lower() == 'no'].copy()

    injury_cols = [
        'Total.Fatal.Injuries',
        'Total.Serious.Injuries',
        'Total.Minor.Injuries',
        'Total.Uninjured'
    ]
    for col in injury_cols:
        aviation[col] = pd.to_numeric(aviation[col], errors='coerce').fillna(0)

    aviation['Total.Onboard.Est'] = aviation[injury_cols].sum(axis=1)
    aviation = aviation[aviation['Total.Onboard.Est'] > 0].copy()
    aviation['Serious.Or.Fatal.Rate'] = (
        aviation['Total.Fatal.Injuries'] + aviation['Total.Serious.Injuries']
    ) / aviation['Total.Onboard.Est']

    aviation['Aircraft.damage'] = aviation['Aircraft.damage'].astype(str).str.strip()
    aviation['Aircraft.damage'] = aviation['Aircraft.damage'].replace({
        'nan': np.nan,
        'Unknown': np.nan,
        'UNKNOWN': np.nan
    })
    aviation['Aircraft.Destroyed'] = aviation['Aircraft.damage'].str.lower().eq('destroyed')

    aviation['Make'] = aviation['Make'].replace({'nan': np.nan, '': np.nan}).str.title()
    aviation['Model'] = aviation['Model'].replace({'nan': np.nan, '': np.nan}).str.upper()
    aviation = aviation.dropna(subset=['Make', 'Model']).copy()

    popular_makes = aviation['Make'].value_counts()[lambda s: s >= 50].index
    aviation = aviation[aviation['Make'].isin(popular_makes)].copy()
    aviation['Plane.Type'] = (aviation['Make'] + ' ' + aviation['Model']).str.replace(r'\s+', ' ', regex=True)

    for col in ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']:
        aviation[col] = aviation[col].astype(str).str.strip().replace({
            '': np.nan,
            'nan': np.nan,
            'UNK': np.nan,
            'UNKNOWN': np.nan,
            'Unknown': np.nan
        })
    aviation['Number.of.Engines'] = pd.to_numeric(aviation['Number.of.Engines'], errors='coerce')

# Make sure metrics are numeric even when loaded from CSV.
for col in ['Total.Onboard.Est', 'Serious.Or.Fatal.Rate', 'Number.of.Engines']:
    aviation[col] = pd.to_numeric(aviation[col], errors='coerce')
aviation['Aircraft.Destroyed'] = aviation['Aircraft.Destroyed'].astype(bool)

aviation = aviation.dropna(subset=['Total.Onboard.Est', 'Serious.Or.Fatal.Rate', 'Make', 'Model', 'Plane.Type']).copy()
aviation = aviation[aviation['Total.Onboard.Est'] > 0].copy()
aviation['Aircraft.Size'] = np.where(
    aviation['Total.Onboard.Est'] <= 20,
    'Small (<=20 onboard)',
    'Large (>20 onboard)'
)

print('Analysis dataset shape:', aviation.shape)
display(aviation[['Event.Date', 'Make', 'Model', 'Plane.Type', 'Total.Onboard.Est',
                  'Serious.Or.Fatal.Rate', 'Aircraft.Destroyed', 'Aircraft.Size']].head())
display(aviation['Aircraft.Size'].value_counts().rename_axis('Aircraft size').reset_index(name='accidents'))


## Explore safety metrics across models/makes
- Remember that the client is interested in separate recommendations for smaller airplanes and larger airplanes. Choose a passenger threshold of 20 and separate the plane types. 

In [ ]:
small_planes = aviation[aviation['Aircraft.Size'] == 'Small (<=20 onboard)'].copy()
large_planes = aviation[aviation['Aircraft.Size'] == 'Large (>20 onboard)'].copy()

MIN_MAKE_ACCIDENTS = 20
MIN_TYPE_ACCIDENTS = 10

def summarize_makes(data, min_n=MIN_MAKE_ACCIDENTS):
    summary = (
        data.groupby('Make')
        .agg(
            accidents=('Make', 'size'),
            mean_serious_fatal_rate=('Serious.Or.Fatal.Rate', 'mean'),
            median_serious_fatal_rate=('Serious.Or.Fatal.Rate', 'median'),
            destroyed_rate=('Aircraft.Destroyed', 'mean'),
            median_onboard=('Total.Onboard.Est', 'median')
        )
        .query('accidents >= @min_n')
        .sort_values(['mean_serious_fatal_rate', 'destroyed_rate', 'accidents'], ascending=[True, True, False])
    )
    return summary

def summarize_plane_types(data, min_n=MIN_TYPE_ACCIDENTS):
    summary = (
        data.groupby('Plane.Type')
        .agg(
            make=('Make', 'first'),
            model=('Model', 'first'),
            accidents=('Plane.Type', 'size'),
            mean_serious_fatal_rate=('Serious.Or.Fatal.Rate', 'mean'),
            median_serious_fatal_rate=('Serious.Or.Fatal.Rate', 'median'),
            destroyed_rate=('Aircraft.Destroyed', 'mean'),
            median_onboard=('Total.Onboard.Est', 'median')
        )
        .query('accidents >= @min_n')
        .sort_values(['mean_serious_fatal_rate', 'destroyed_rate', 'accidents'], ascending=[True, True, False])
    )
    return summary

small_make_summary = summarize_makes(small_planes)
large_make_summary = summarize_makes(large_planes)
small_type_summary = summarize_plane_types(small_planes)
large_type_summary = summarize_plane_types(large_planes)

print('Small-aircraft accidents:', len(small_planes))
print('Large-aircraft accidents:', len(large_planes))
print('Small makes meeting threshold:', small_make_summary.shape[0])
print('Large makes meeting threshold:', large_make_summary.shape[0])


#### Analyzing Makes

Explore the human injury risk profile for small and larger Makes:
- choose the 15 makes for each group possessing the lowest mean fatal/seriously injured fraction
- plot the mean fatal/seriously injured fraction for each of these subgroups side-by-side

In [ ]:
small_top15 = small_make_summary.head(15).reset_index()
large_top15 = large_make_summary.head(15).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

sns.barplot(
    data=small_top15,
    y='Make',
    x='mean_serious_fatal_rate',
    ax=axes[0],
    color='#4C78A8'
)
axes[0].set_title('Small aircraft: lowest serious/fatal injury rates')
axes[0].set_xlabel('Mean serious or fatal injury fraction')
axes[0].set_ylabel('Make')

sns.barplot(
    data=large_top15,
    y='Make',
    x='mean_serious_fatal_rate',
    ax=axes[1],
    color='#F58518'
)
axes[1].set_title('Large aircraft: lowest serious/fatal injury rates')
axes[1].set_xlabel('Mean serious or fatal injury fraction')
axes[1].set_ylabel('Make')

plt.tight_layout()
plt.show()

display(small_top15)
display(large_top15)


**Distribution of injury rates: small makes**

Use a violinplot to look at the distribution of the fraction of passengers serious/fatally injured for small airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
small_low10_makes = small_make_summary.head(10).index
small_low10_data = small_planes[small_planes['Make'].isin(small_low10_makes)].copy()

plt.figure(figsize=(14, 7))
sns.violinplot(
    data=small_low10_data,
    x='Serious.Or.Fatal.Rate',
    y='Make',
    order=small_low10_makes,
    inner='quartile',
    cut=0,
    color='#72B7B2'
)
plt.title('Distribution of serious/fatal injury fraction for low-risk small-aircraft makes')
plt.xlabel('Serious or fatal injury fraction')
plt.ylabel('Make')
plt.tight_layout()
plt.show()


**Distribution of injury rates: large makes**

Use a stripplot to look at the distribution of the fraction of passengers serious/fatally injured for large airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
large_low10_makes = large_make_summary.head(10).index
large_low10_data = large_planes[large_planes['Make'].isin(large_low10_makes)].copy()

plt.figure(figsize=(14, 6))
sns.stripplot(
    data=large_low10_data,
    x='Serious.Or.Fatal.Rate',
    y='Make',
    order=large_low10_makes,
    alpha=0.55,
    jitter=0.25,
    color='#E45756'
)
plt.title('Distribution of serious/fatal injury fraction for low-risk large-aircraft makes')
plt.xlabel('Serious or fatal injury fraction')
plt.ylabel('Make')
plt.tight_layout()
plt.show()


**Evaluate the rate of aircraft destruction for both small and large aircraft by Make.** 

Sort your results and keep the lowest 15.

In [ ]:
small_destroyed_top15 = small_make_summary.sort_values(['destroyed_rate', 'mean_serious_fatal_rate']).head(15).reset_index()
large_destroyed_top15 = large_make_summary.sort_values(['destroyed_rate', 'mean_serious_fatal_rate']).head(15).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

sns.barplot(data=small_destroyed_top15, y='Make', x='destroyed_rate', ax=axes[0], color='#54A24B')
axes[0].set_title('Small aircraft: lowest destruction rates')
axes[0].set_xlabel('Destroyed fraction')
axes[0].set_ylabel('Make')

sns.barplot(data=large_destroyed_top15, y='Make', x='destroyed_rate', ax=axes[1], color='#B279A2')
axes[1].set_title('Large aircraft: lowest destruction rates')
axes[1].set_xlabel('Destroyed fraction')
axes[1].set_ylabel('Make')

plt.tight_layout()
plt.show()

display(small_destroyed_top15)
display(large_destroyed_top15)


#### Provide a short discussion on your findings for your summary statistics and plots:
- Make any recommendations for Makes here based off of the destroyed fraction and fraction fatally/seriously injured
- Comment on the calculated statistics and any corresponding distributions you have visualized.

**Findings and make-level recommendations**

The make-level comparisons only use makes with at least 20 accident records inside each aircraft-size group. This reduces the chance of recommending a make because of a single unusually mild incident.

For small aircraft, **Waco**, **Maule**, **Aviat Aircraft Inc**, **Let**, **Boeing Stearman**, and **Stinson** stand out because they combine relatively low mean serious/fatal injury fractions with low-to-moderate destruction rates. **Grumman-Schweizer** has a low injury rate, but its destruction rate is materially higher, so it should be treated as less attractive from an insurer's hull-loss perspective.

For larger passenger aircraft, the strongest manufacturer-level candidates are **McDonnell Douglas**, **Embraer**, **Boeing**, and **Airbus Industrie**. McDonnell Douglas has the lowest mean serious/fatal injury fraction in this filtered group, while Embraer has the lowest destruction rate among the large-aircraft makes that meet the sample threshold. These results should be read as accident-outcome comparisons, not as accident-frequency comparisons, because the dataset only contains accidents and incidents rather than total flights or flight hours.

The distribution plots show why averages alone are not enough: many makes have a large cluster of zero-injury accidents plus a small number of severe events. Recommendations therefore weigh both the mean injury fraction and the aircraft destruction fraction instead of relying on a single metric.


### Analyze plane types
- plot the mean fatal/seriously injured fraction for both small and larger planes 
- also provide a distributional plot of your choice for the fatal/seriously injured fraction by airplane type (stripplot, violin, etc)  
- filter ensuring that you have at least ten individual examples in each model/make to average over

**Larger planes**

In [ ]:
large_type_top15 = large_type_summary.head(15).reset_index()

plt.figure(figsize=(14, 7))
sns.barplot(
    data=large_type_top15,
    y='Plane.Type',
    x='mean_serious_fatal_rate',
    color='#4C78A8'
)
plt.title('Large aircraft models with lowest mean serious/fatal injury fraction')
plt.xlabel('Mean serious or fatal injury fraction')
plt.ylabel('Plane type')
plt.tight_layout()
plt.show()

top_large_types = large_type_top15['Plane.Type']
plt.figure(figsize=(14, 7))
sns.stripplot(
    data=large_planes[large_planes['Plane.Type'].isin(top_large_types)],
    y='Plane.Type',
    x='Serious.Or.Fatal.Rate',
    order=top_large_types,
    alpha=0.65,
    jitter=0.25,
    color='#F58518'
)
plt.title('Accident-level injury-rate distribution for recommended large aircraft models')
plt.xlabel('Serious or fatal injury fraction')
plt.ylabel('Plane type')
plt.tight_layout()
plt.show()

display(large_type_top15)


**Smaller planes**
- for smaller planes, limit your plotted results to the makes with the 10 lowest mean serious/fatal injury fractions

In [ ]:
small_type_top15 = small_type_summary.head(15).reset_index()
small_type_low_makes = small_make_summary.head(10).index
small_type_top_by_low_makes = (
    small_type_summary[small_type_summary['make'].isin(small_type_low_makes)]
    .head(15)
    .reset_index()
)

plt.figure(figsize=(14, 7))
sns.barplot(
    data=small_type_top_by_low_makes,
    y='Plane.Type',
    x='mean_serious_fatal_rate',
    color='#54A24B'
)
plt.title('Small aircraft models with lowest mean serious/fatal injury fraction')
plt.xlabel('Mean serious or fatal injury fraction')
plt.ylabel('Plane type')
plt.tight_layout()
plt.show()

top_small_types = small_type_top_by_low_makes['Plane.Type']
plt.figure(figsize=(14, 7))
sns.violinplot(
    data=small_planes[small_planes['Plane.Type'].isin(top_small_types)],
    y='Plane.Type',
    x='Serious.Or.Fatal.Rate',
    order=top_small_types,
    inner='quartile',
    cut=0,
    color='#72B7B2'
)
plt.title('Accident-level injury-rate distribution for recommended small aircraft models')
plt.xlabel('Serious or fatal injury fraction')
plt.ylabel('Plane type')
plt.tight_layout()
plt.show()

display(small_type_top_by_low_makes)


### Discussion of Specific Airplane Types
- Discuss what you have found above regarding passenger fraction seriously/ both small and large airplane models.

**Specific airplane type recommendations**

For larger passenger aircraft, the most attractive model-level candidates in this accident-outcome dataset include **Boeing 717-200**, **Boeing 757-232**, **Boeing 757-222**, **McDonnell Douglas MD-88**, **Embraer EMB-145LR**, and **McDonnell Douglas MD-11**. These models meet the minimum sample threshold and show low mean serious/fatal injury fractions. Several of them also have zero observed destroyed-aircraft outcomes in the filtered accident sample, which strengthens their case for the client's combined injury-and-hull-loss criteria.

For smaller aircraft, **Piper PA-18A 150**, **Diamond Aircraft Ind Inc DA 20 C1**, **Bell 47D-1**, **Maule MX-7-235**, **Air Tractor AT 602**, and **Piper PA-18-160** are notable candidates. The very lowest small-aircraft injury rates often occur in one- or two-person aircraft, so the recommendations should be read within the small-aircraft segment only rather than compared directly to larger aircraft.

A key limitation is that this analysis measures severity conditional on an accident being recorded. It does not measure the probability of an accident per flight hour, departure, or aircraft in service.


### Exploring Other Variables
- Investigate how other variables effect aircraft damage and injury. You must choose **two** factors out of the following but are free to analyze more:

- Weather Condition
- Engine Type
- Number of Engines
- Phase of Flight
- Purpose of Flight

For each factor provide a discussion explaining your analysis with appropriate visualization / data summaries and interpreting your findings.

In [ ]:
def summarize_factor(data, factor, min_n=100):
    return (
        data.dropna(subset=[factor])
        .groupby(factor)
        .agg(
            accidents=(factor, 'size'),
            mean_serious_fatal_rate=('Serious.Or.Fatal.Rate', 'mean'),
            median_serious_fatal_rate=('Serious.Or.Fatal.Rate', 'median'),
            destroyed_rate=('Aircraft.Destroyed', 'mean'),
            median_onboard=('Total.Onboard.Est', 'median')
        )
        .query('accidents >= @min_n')
        .sort_values('mean_serious_fatal_rate')
    )

weather_summary = summarize_factor(aviation, 'Weather.Condition', min_n=100)
phase_summary = summarize_factor(aviation, 'Broad.phase.of.flight', min_n=100)
engine_summary = summarize_factor(aviation, 'Engine.Type', min_n=100)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(
    data=weather_summary.reset_index(),
    x='Weather.Condition',
    y='mean_serious_fatal_rate',
    ax=axes[0],
    color='#4C78A8'
)
axes[0].set_title('Mean serious/fatal injury fraction by weather')
axes[0].set_xlabel('Weather condition')
axes[0].set_ylabel('Mean serious/fatal injury fraction')

sns.barplot(
    data=weather_summary.reset_index(),
    x='Weather.Condition',
    y='destroyed_rate',
    ax=axes[1],
    color='#E45756'
)
axes[1].set_title('Destroyed fraction by weather')
axes[1].set_xlabel('Weather condition')
axes[1].set_ylabel('Destroyed fraction')

plt.tight_layout()
plt.show()

display(weather_summary)

phase_plot = phase_summary.reset_index().sort_values('mean_serious_fatal_rate')
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(
    data=phase_plot,
    y='Broad.phase.of.flight',
    x='mean_serious_fatal_rate',
    ax=axes[0],
    color='#54A24B'
)
axes[0].set_title('Mean serious/fatal injury fraction by phase')
axes[0].set_xlabel('Mean serious/fatal injury fraction')
axes[0].set_ylabel('Broad phase of flight')

sns.barplot(
    data=phase_plot,
    y='Broad.phase.of.flight',
    x='destroyed_rate',
    ax=axes[1],
    color='#B279A2'
)
axes[1].set_title('Destroyed fraction by phase')
axes[1].set_xlabel('Destroyed fraction')
axes[1].set_ylabel('Broad phase of flight')

plt.tight_layout()
plt.show()

display(phase_summary)

engine_plot = engine_summary.reset_index().sort_values('mean_serious_fatal_rate')
plt.figure(figsize=(12, 5))
sns.barplot(
    data=engine_plot,
    x='Engine.Type',
    y='mean_serious_fatal_rate',
    color='#F58518'
)
plt.title('Mean serious/fatal injury fraction by engine type')
plt.xlabel('Engine type')
plt.ylabel('Mean serious/fatal injury fraction')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

display(engine_summary)

print('Interpretation:')
print('- Weather: IMC accidents have a much higher mean serious/fatal injury fraction and destruction rate than VMC accidents.')
print('- Phase of flight: taxi and landing accidents tend to have the lowest severity; climb, cruise, approach, and maneuvering have substantially higher injury and destruction rates.')
print('- Engine type is partly confounded with aircraft size and mission, but turbo-fan aircraft show lower conditional injury severity than the other high-sample engine groups in this accident dataset.')
